# GradientAscent · UPI Fraud Rings

**Team:** GradientAscent · **Member:** Ameya Borkar

**Problem.** Detect which of 260,582 October UPI payments belong to organised fraud rings, and name the scam behind each one (legit plus 11 fraud types), even though most fraud looks like an ordinary payment and the training labels are incomplete. Scored by 12-class macro-F1.

**This notebook rebuilds our best upload, `stack-burst12-sim45.csv` (public macro-F1 0.5556), byte for byte**, from the code frozen at tag `v1.0`:
1. get the code and install it;
2. point it at the organisers' data;
3. run one command: label-free features → six member models on month folds → mix, decoder and rules → the October file;
4. check the sha256 and read the scores.

Expected sha256: `c278490094706e156dc4f4d573989fafda24de896f3a7ec147fb5fc886eb2efc`. About 90 minutes on one GPU from an empty cache.

## How the system works

| Layer | What it does |
|---|---|
| Label-free evidence | Rebuilds each fraud routine from how payments connect (accounts, phones, receivers, time): layering chains, mule fan-ins, bot bursts, balance drains, collect requests and baits, refund and lending chains, merchant rings |
| Device-number counter | Fresh phones are numbered in creation order, and the fraud routines write their phones class by class. An 11-state HMM segments that block, so a phone's position certifies ordinary-looking fraud rows and gives their type |
| Six members | Five boosted cascades (XGBoost, LightGBM, CatBoost, two XGBoost on event-cleaned labels) and a TabM network, trained on whole months as folds |
| Decision | The trees set the scam type, TabM and the trees set fraud-or-not; an expected-macro-F1 decoder per month, fraud probabilities divided by 0.875 for the unreported fraud in training |
| Rules | Counter-anchored look-alike templates, operator-role bait rules and capped base-rate rules |

## 1. Get the code (frozen at tag `v1.0`) and install it

In [ ]:
!git clone --branch v1.0 --depth 1 https://github.com/AmeyaBorkar/gradient-ascent.git
%cd gradient-ascent
!pip install -q -e ".[dev,nn]"

Check the install with the unit tests (synthetic data only, under a minute):

In [ ]:
!python -m pytest -q tests

## 2. Point it at the data

Put the organisers' files (`train.csv`, `test.csv`, `accounts.csv`, `merchants.csv`, `sample_submission.csv`) in one folder and set `DATA` to it. The cache and output folders are created here.

In [ ]:
import os
DATA = "/content/data"            # folder with the organisers' five CSV files
os.environ["RINGTRACE_DATA"] = DATA
os.environ["RINGTRACE_CACHE"] = os.path.abspath("cache")
os.environ["RINGTRACE_OUT"] = os.path.abspath("outputs")
os.environ["RINGTRACE_THREADS"] = "8"
for f in ["train.csv", "test.csv", "accounts.csv", "merchants.csv", "sample_submission.csv"]:
    print(f, "found" if os.path.exists(os.path.join(DATA, f)) else "MISSING")

## 3. Rebuild the best upload (one command)

The script builds every label-free feature family (cached by content hash), freezes the cleaned labels used for scoring, trains the six members on month folds (three seeds each; a member whose arrays already exist is kept), then mixes, decodes and applies the rules, and validates the file against `sample_submission.csv`.

In [ ]:
!bash scripts/build_final.sh recipes/stack/stack-burst12-sim45.json out

## 4. Check the result

In [ ]:
import hashlib, json
path = "out/stack-burst12-sim45.csv"
sha = hashlib.sha256(open(path, "rb").read()).hexdigest()
print(sha)
print("byte-identical to the upload:", sha == "c278490094706e156dc4f4d573989fafda24de896f3a7ec147fb5fc886eb2efc")

In [ ]:
summary = json.load(open("out/stack-burst12-sim45.json"))
print("October calls per class:", summary["calls"])
print("October rows each base-rate rule changed:", {k: v["october"] for k, v in summary["base_rate_changes"].items()})
print("leave-one-month-out macro-F1:", json.dumps(summary["scores"], indent=1)[:1500])

## Notes

- **Validation.** Every model change was judged on held-out months (given, cleaned and generator-only labels, with a paired event bootstrap) and on September thinned to October's density with the device counter rebuilt. The last two steps of this file (the cuts of the fresh-SIM burst rule and one removed border row) were settled with public leaderboard reads, as the repository states.
- **Determinism.** Seeds are fixed. TabM's arrays can differ in the last bits on another GPU model and CatBoost on the GPU is not bit-reproducible; neither moved a call in our rebuilds.
- **More.** The full report is `docs/report/GradientAscent_Documentation.pdf`; every finding is indexed in `docs/README.md`.